In [1]:
import re
import os
import numpy as np
import pandas as pd
from pathlib import Path

In [2]:
ROOT_DIR = Path("/mnt/isilon/bgdlab_hbcd/projects/macedo_wm_exposome/macedo_wm_exposome")
INPUT_DATA_DIR = ROOT_DIR / "input_data"

In [3]:
DATA_PATH = Path("/mnt/isilon/bgdlab_hbcd/projects/meisler_abcd_dmri_new/data/raw_data/merged_data_meisler_analyses.parquet")

data = pd.read_parquet(DATA_PATH, engine="fastparquet")

# only include baseline data
data = data[data["session_id"] == "ses-00A"]

print(data.shape)

(8402, 5499)


data.shape (steven's prior data)-> (8768, 4325)

data.shape (with most up to date parquet) -> (8402, 5499)

In [4]:
# Filter to only keep specific columns along with all tracts/metrics
def filter_columns_by_tracts_and_metrics(columns):
    selected_cols = ["subject_session", "subject_id", "session_id", "age", "site", "sex", "t1post_dwi_contrast", 
                     "batch_device_software", "eTIV", "scanner_model", "scanner_manufacturer"]
    for col in columns:
        parts = col.split('_')
        if parts[0] == 'bundle': selected_cols.append(col) # white matter feature columns
    return selected_cols

selected_columns = filter_columns_by_tracts_and_metrics(data.columns)
data_reduced = data[selected_columns].copy()
print(data_reduced.shape)

(8402, 5405)


In [5]:
data_reduced["batch_device_software"].value_counts()

batch_device_software
anon0bd7.Siemens_VE11B    717
anonefd2.Siemens_VE11C    487
anonfaf6.Siemens_VE11B    474
anon3eba.Siemens_VE11C    446
anon8928.Siemens_VE11C    416
anon2565.Siemens_VE11B    399
anonbe03.Siemens_VE11B    398
anon3255.GE_DV25          387
anonbcc2.GE_DV25          387
anonf61b.GE_DV25          365
anondc7b.Siemens_VE11C    315
anonc805.Siemens_VE11C    304
anond450.Philips_5.3      301
anon224d.Siemens_VE11C    298
anon7a3b.Siemens_VE11B    292
anon02a4.Siemens_VE11C    260
anonb2d4.Philips_5.3      247
anon1364.Philips_5.3      246
anon239d.GE_DV25          216
anon241f.Siemens_VE11C    187
anon0982.GE_DV26          179
anon7353.Siemens_VE11C    161
anon12da.GE_DV26          148
anon9d57.GE_DV26          107
anon2b69.GE_DV25          103
anon0bd7.Siemens_VE11C     91
anon12da.GE_DV25           88
anon0982.GE_DV25           86
anona429.Siemens_VE11C     62
anon3255.GE_DV26           47
anon6ecd.Siemens_VE11C     42
anonbcc2.GE_DV26           32
anon2b69.GE_DV26  

data_reduced.shape -> (8402, 5405)

### ABCD Reproducible Matched Samples
Demographically matched samples.
We will perform 2-fold cross-validation with the matched samples serving as either the training or testing set in different folds.

In [5]:
def clean_coded_column(df, col, valid_range=None, missing_codes=(777, 999)):
    s = pd.to_numeric(df[col])               # converts numeric strings to numerics
    s = s.replace({code: np.nan for code in missing_codes})  # set missing codes to NaN
    lo, hi = valid_range
    s = s.where((s >= lo) & (s <= hi), np.nan) # if not in valid range, set to NaN
    df[col] = s
    return df

In [6]:
main_analysis = False
edu_sensitivity = False # sensitivity analysis, looking at income and parental education rather than exposome score
cognition_analysis = True 
if (main_analysis + edu_sensitivity + cognition_analysis != 1): 
    print("PLEASE SELECT ONLY ONE CASE. ONE OPTION SHOULD BE TRUE, THE REST SHOULD BE FALSE")

In [7]:
# area deprivation index
adi_path = Path("/mnt/isilon/bgdlab_hbcd/shared_data/ABCD/LASSO_tabular/dairc/rawdata/phenotype/le_l_adi.parquet")
adi = pd.read_parquet(adi_path)
adi = adi[adi["session_id"] == "ses-00A"]
print(adi.shape)
adi.head(3)

(11214, 59)


,participant_id,session_id,le_l_adi__addr1__crowding_prcnt,le_l_adi__addr1__eduhs_prcnt,le_l_adi__addr1__edulths_prcnt,le_l_adi__addr1__homeowner_prcnt,le_l_adi__addr1__homevalue_med,le_l_adi__addr1__income__disparity_score,le_l_adi__addr1__income_med,le_l_adi__addr1__mortgage_med,...,le_l_adi__addr3__nocar_prcnt,le_l_adi__addr3__nophone_prcnt,le_l_adi__addr3__noplumbing_prcnt,le_l_adi__addr3__poverty__below138_prcnt,le_l_adi__addr3__poverty_prcnt,le_l_adi__addr3__rent_med,le_l_adi__addr3__singlehousehold_prcnt,le_l_adi__addr3__unemployment_prcnt,le_l_adi__addr3__workwhitecollar_prcnt,le_l_adi__addr3_wsum
0,sub-005V6D2C,ses-00A,2.440678,87.37314,4.861931,16.290290,271100.0,2.946340,45609.0,1804.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,sub-007W6H7B,ses-00A,2.930622,94.81628,3.559711,26.772322,897800.0,3.567694,129961.0,1534.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,sub-00BD7VDC,ses-00A,0.000000,96.58959,0.635838,73.894310,149600.0,1.344857,84150.0,1165.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [8]:
participants = pd.read_csv(INPUT_DATA_DIR / "participants.tsv", sep="\t")
participants = participants.loc[participants["session_id"] == "ses-baselineYear1Arm1"].copy()  # baseline arm only

# make subject id formatting consistent across datasets
participants.loc[:, "subject_id_clean"] = participants["participant_id"].str.replace(r"^sub-NDARINV", "", regex=True)
data_reduced.loc[:, "subject_id_clean"] = data_reduced["subject_id"].str.replace(r"^sub-", "", regex=True)
adi.loc[:, "subject_id_clean"] = adi["participant_id"].astype(str).str.replace(r"^sub-", "", regex=True)

# keep only the ADI variable we want
adi_lookup = (adi.loc[:, ["subject_id_clean", "le_l_adi__addr1__national_prcnt"]]
    .drop_duplicates(subset=["subject_id_clean"])
    .set_index("subject_id_clean"))

if not edu_sensitivity:
    matched_info = participants.loc[:, ["subject_id_clean", "matched_group"]].drop_duplicates()
    data_reduced.loc[:, "matched_group"] = data_reduced["subject_id_clean"].map(matched_info.set_index("subject_id_clean")["matched_group"])
else:
    participants = clean_coded_column(participants, col="parental_education",  valid_range=(0, 21), missing_codes=(777, 999))
    participants = clean_coded_column(participants, col="income", valid_range=(1, 9), missing_codes=(777, 999))

    # merge ADI into participants first
    participants = participants.merge(adi_lookup.reset_index(), on="subject_id_clean", how="left")

    # drop participants with missing required sensitivity vars
    participants_clean = participants.dropna(subset=["parental_education", "income", "le_l_adi__addr1__national_prcnt"]).copy()

    cols_to_transfer = ["matched_group", "parental_education", "income", "le_l_adi__addr1__national_prcnt"]

    lookup = (participants_clean.loc[:, ["subject_id_clean"] + cols_to_transfer].drop_duplicates(subset=["subject_id_clean"])
        .set_index("subject_id_clean"))

    for col in cols_to_transfer:
        data_reduced[col] = data_reduced["subject_id_clean"].map(lookup[col])

print(data_reduced["matched_group"].value_counts(dropna=False))
print(data_reduced.shape)
data_reduced.head(3)

matched_group
2.0      4101
1.0      4082
3.0       216
NaN         2
888.0       1
Name: count, dtype: int64
(8402, 5407)


,subject_session,subject_id,session_id,age,site,sex,t1post_dwi_contrast,batch_device_software,eTIV,scanner_model,...,bundle_Cerebellum_InferiorCerebellarPeduncleR_NODDI_isovf_median,bundle_Cerebellum_InferiorCerebellarPeduncleR_NODDI_od_median,bundle_Cerebellum_InferiorCerebellarPeduncleL_NODDI_icvf_median,bundle_Cerebellum_InferiorCerebellarPeduncleL_NODDI_isovf_median,bundle_Cerebellum_InferiorCerebellarPeduncleL_NODDI_od_median,bundle_Cerebellum_CerebellumL_NODDI_icvf_median,bundle_Cerebellum_CerebellumL_NODDI_isovf_median,bundle_Cerebellum_CerebellumL_NODDI_od_median,subject_id_clean,matched_group
0,sub-005V6D2C_ses-00A,sub-005V6D2C,ses-00A,10.117808,10,2.0,1.756409,anon9d57.GE_DV26,1.255721e+06,DISCOVERY MR750,...,0.124215,0.420602,0.542971,0.118243,0.389446,0.504545,0.123380,0.491271,005V6D2C,2.0
3,sub-00CY2MDM_ses-00A,sub-00CY2MDM,ses-00A,10.887671,20,1.0,1.816654,anon224d.Siemens_VE11C,1.516595e+06,Prisma,...,0.119992,0.390373,0.553554,0.106922,0.403003,0.522555,0.112646,0.447257,00CY2MDM,1.0
7,sub-00HEV6HB_ses-00A,sub-00HEV6HB,ses-00A,10.356164,12,1.0,1.645437,anonefd2.Siemens_VE11C,1.406408e+06,Prisma_fit,...,0.118988,0.371245,0.562124,0.106032,0.383102,0.504545,0.116645,0.440266,00HEV6HB,1.0


main analysis:
* shape = (8402, 5407)
* matched_group:  
    * 2.0:      4101
    * 1.0:      4082
    * 3.0:       216
    * NaN:         2
    * 888.0:       1

edu_sens:
* shape = (8402, 5410)
* matched_group
    * 2.0    3126
    * 1.0    3101
    * NaN    2002
    * 3.0     173

cognition:
* shape = (8402, 5407)
* matched_group
    * 2.0      4101
    * 1.0      4082
    * 3.0       216
    * NaN         2
    * 888.0       1

In [9]:
data_reduced = data_reduced.dropna(subset=["matched_group"])
data_reduced = data_reduced[data_reduced["matched_group"] != 888.0]
data_reduced = data_reduced[data_reduced["matched_group"] != 3]

print(data_reduced["matched_group"].value_counts(dropna=False))

matched_group
2.0    4101
1.0    4082
Name: count, dtype: int64


In [10]:
data_reduced.shape

(8183, 5407)

main_analysis: (8183, 5407)
edu sens: (6227, 5410)
cognition: (8183, 5407)

### Exposome scores

In [11]:
exposome = pd.read_csv(INPUT_DATA_DIR / "abcd_longitudinal_factor_scores_7f_bifactor.csv")
exposome['subject_id_clean'] = exposome['ID'].str.replace(r'^NDAR_INV', '', regex=True)
print(exposome.shape)

(33372, 13)


In [12]:
exposome_filtered = exposome[exposome["Timepoint"] == "baseline_year_1_arm_1"]
exposome_filtered = exposome_filtered[["General_SES", "School", "Family_Values", "Family_Turmoil", 
                                       "Dense_Urban_Poverty", "Extracurriculars", "Screen_Time", "subject_id_clean"]]
print(exposome_filtered.shape)

(11874, 8)


### Cognition scores

In [13]:
cognition_data = pd.read_csv(INPUT_DATA_DIR / "cognition_data.csv")
cognition_data_filtered = cognition_data[cognition_data["event_name"] == "baseline_year_1_arm_1"]
cognition_core = cognition_data_filtered[["src_subject_id", "neurocog_pc1.bl", "neurocog_pc2.bl", "neurocog_pc3.bl"]]
cognition_core.loc[:, 'subject_id_clean'] = cognition_core['src_subject_id'].str.replace(r'^NDAR_INV', '', regex=True)
print(cognition_core.shape)
cognition_core.head(1)

(11878, 5)


/tmp/ipykernel_2453641/1127279331.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  cognition_core.loc[:, 'subject_id_clean'] = cognition_core['src_subject_id'].str.replace(r'^NDAR_INV', '', regex=True)


,src_subject_id,neurocog_pc1.bl,neurocog_pc2.bl,neurocog_pc3.bl,subject_id_clean
1,NDAR_INV003RTV85,0.278102,0.340003,0.237957,003RTV85


### Drop bundles and metrics that we are not interested in

In [14]:
bundles_to_remove = [
    "bundle_ProjectionBrainstem_DentatorubrothalamicTractL",
    "bundle_Projection_BrainstemDentatorubrothalamicTractR",
    "bundle_Commissure_AnteriorCommissure",
    "bundle_ProjectionBrainstem_CorticobulbarTractL",
    "bundle_ProjectionBrainstem_CorticobulbarTractR"
]

metrics_to_remove = ["number_of_tracts", "NODDI_directions"]

# --- Initial merge ---
if edu_sensitivity:
    merged_exposome = data_reduced.copy()
    print(f"[INIT] Using data_reduced only → shape: {merged_exposome.shape}")
else:
    merged_exposome = exposome_filtered.merge(
        data_reduced, on="subject_id_clean", how="outer"
    )
    print(f"[INIT] After merging exposome_filtered + data_reduced → shape: {merged_exposome.shape}")

# --- Cognition merge ---
if cognition_analysis:
    merged_exposome = merged_exposome.merge(
        cognition_core, on="subject_id_clean", how="outer"
    )
    print(f"[COGNITION MERGE] After adding cognition_core → shape: {merged_exposome.shape}")

# --- Drop bundle columns ---
cols_to_drop = [
    col for col in merged_exposome.columns
    if any(bundle in col for bundle in bundles_to_remove)
]
print(f"[DROP BUNDLES] Removing {len(cols_to_drop)} columns")
merged_exposome = merged_exposome.drop(columns=cols_to_drop)
print(f"[DROP BUNDLES] New shape → {merged_exposome.shape}")

# --- Drop metric columns ---
cols_to_drop = [
    col for col in merged_exposome.columns
    if any(metric in col for metric in metrics_to_remove)
]
print(f"[DROP METRICS] Removing {len(cols_to_drop)} columns")
merged_exposome = merged_exposome.drop(columns=cols_to_drop)
print(f"[DROP METRICS] New shape → {merged_exposome.shape}")

# --- Drop NA ---
before_rows = merged_exposome.shape[0]
merged_exposome = merged_exposome.dropna()
after_rows = merged_exposome.shape[0]

print(f"[DROP NA] Rows before: {before_rows}, after: {after_rows}, removed: {before_rows - after_rows}")
print(f"[FINAL] Final shape → {merged_exposome.shape}")

# Preview
merged_exposome.head(2)

[INIT] After merging exposome_filtered + data_reduced → shape: (11876, 5414)
[COGNITION MERGE] After adding cognition_core → shape: (11878, 5418)
[DROP BUNDLES] Removing 0 columns
[DROP BUNDLES] New shape → (11878, 5418)
[DROP METRICS] Removing 62 columns
[DROP METRICS] New shape → (11878, 5356)
[DROP NA] Rows before: 11878, after: 7638, removed: 4240
[FINAL] Final shape → (7638, 5356)


,General_SES,School,Family_Values,Family_Turmoil,Dense_Urban_Poverty,Extracurriculars,Screen_Time,subject_id_clean,subject_session,subject_id,...,bundle_Cerebellum_InferiorCerebellarPeduncleL_NODDI_isovf_median,bundle_Cerebellum_InferiorCerebellarPeduncleL_NODDI_od_median,bundle_Cerebellum_CerebellumL_NODDI_icvf_median,bundle_Cerebellum_CerebellumL_NODDI_isovf_median,bundle_Cerebellum_CerebellumL_NODDI_od_median,matched_group,src_subject_id,neurocog_pc1.bl,neurocog_pc2.bl,neurocog_pc3.bl
1,-0.975790,1.003044,0.429337,-1.085106,1.752441,-0.945097,-1.305787,005V6D2C,sub-005V6D2C_ses-00A,sub-005V6D2C,...,0.118243,0.389446,0.504545,0.123380,0.491271,2.0,NDAR_INV005V6D2C,-0.930358,0.070785,-0.047210
4,-0.782426,0.187079,0.328957,1.765782,0.036967,0.575135,-0.906260,00CY2MDM,sub-00CY2MDM_ses-00A,sub-00CY2MDM,...,0.106922,0.403003,0.522555,0.112646,0.447257,1.0,NDAR_INV00CY2MDM,0.171688,0.452998,-1.127686


**main_analysis**
* [INIT] After merging exposome_filtered + data_reduced → shape: (11876, 5414)
* [DROP BUNDLES] Removing 0 columns
* [DROP BUNDLES] New shape → (11876, 5414)
* [DROP METRICS] Removing 62 columns
* [DROP METRICS] New shape → (11876, 5352)
* [DROP NA] Rows before: 11876, after: 8175, removed: 3701
* [FINAL] Final shape → (8175, 5352)

**edu_sensitivity**
* [INIT] Using data_reduced only → shape: (6227, 5410)
* [DROP BUNDLES] Removing 0 columns
* [DROP BUNDLES] New shape → (6227, 5410)
* [DROP METRICS] Removing 62 columns
* [DROP METRICS] New shape → (6227, 5348)
* [DROP NA] Rows before: 6227, after: 6222, removed: 5
* [FINAL] Final shape → (6222, 5348)

**cognition_analysis**
* [INIT] After merging exposome_filtered + data_reduced → shape: (11876, 5414)
* [COGNITION MERGE] After adding cognition_core → shape: (11878, 5418)
* [DROP BUNDLES] Removing 0 columns
* [DROP BUNDLES] New shape → (11878, 5418)
* [DROP METRICS] Removing 62 columns
* [DROP METRICS] New shape → (11878, 5356)
* [DROP NA] Rows before: 11878, after: 7638, removed: 4240
* [FINAL] Final shape → (7638, 5356)

In [15]:
OUTPUT_DIR = ROOT_DIR / "output_data" / "cleaned"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)  # create if it doesn't exist

if edu_sensitivity:
    output_file = OUTPUT_DIR / "parental_edu_income_sensitivity_df.csv"
elif cognition_analysis:
    output_file = OUTPUT_DIR / "exposome_FINAL_cognition_11_10.csv"
else:
    output_file = OUTPUT_DIR / "exposome_FINAL_11_3.csv"

merged_exposome.to_csv(output_file, index=False)

print(f"[SAVE] File saved to: {output_file}")

[SAVE] File saved to: /mnt/isilon/bgdlab_hbcd/projects/macedo_wm_exposome/macedo_wm_exposome/output_data/cleaned/exposome_FINAL_cognition_11_10.csv
